# CrashSignal — Walk-Forward Validation (Regime Shift Fix)

**Problem in v3:** Train F1=0.97, Val F1=0.38
**Diagnosis:** The 2020-2024 validation period contains market regimes (0% rates, pandemic volatility, fastest hike cycle) the model never saw in the 1994-2018 training period.
**Fix:** Walk-Forward Validation. Train on everything up to year N, validate on year N+1. This exposes the model to all market environments in validation and produces an honest Out-Of-Fold (OOF) F1 score.

| Fold | Train Period | Val Year |
|------|--------------|----------|
| 1    | 1994 - 2015  | 2016     |
| 2    | 1994 - 2016  | 2017     |
| 3    | 1994 - 2017  | 2018     |
| ...  | ...          | ...      |
| 9    | 1994 - 2023  | 2024     |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import os, json, joblib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    f1_score, roc_auc_score,
    classification_report, confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm

warnings.filterwarnings("ignore")
plt.style.use("dark_background")

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU. Enable in Settings -> Accelerator")

In [ ]:
# ── Data ──────────────────────────────────────────
CV_START_YEAR = 2016
CV_END_YEAR   = 2024
WINDOW_SIZE   = 60
CLIP_LOW      = 0.02
CLIP_HIGH     = 0.98

# ── Model — REDUCED CAPACITY ──────────────────────
HIDDEN_DIM    = 64       # was 128
LSTM_LAYERS   = 1        # was 2
NUM_HEADS     = 4
NUM_CLASSES   = 3
DROPOUT       = 0.5      # was 0.3

# ── Loss ──────────────────────────────────────────
ALPHA         = 0.5      # CE weight
BETA          = 0.3      # ordinal weight
GAMMA         = 0.2      # smoothness weight
LABEL_SMOOTH  = 0.1      # was 0.0

# ── Training — STRONGER REGULARIZATION ────────────
LR            = 1e-4     # was 3e-4
WEIGHT_DECAY  = 5e-3     # was 1e-3
BATCH_SIZE    = 128      # was 64
NUM_EPOCHS    = 80
PATIENCE      = 20
GRAD_CLIP     = 0.5      # was 1.0
COSINE_TMAX   = 80
COSINE_EMIN   = 1e-5

# ── Paths ─────────────────────────────────────────
DATA_PATH     = "master_df.parquet"
FEAT_PATH     = "feature_columns.json"
CRISIS_PATH   = "crisis_periods.json"
OUT_DIR       = "/kaggle/working"

print("Walk-Forward Hyperparameters loaded.")
print(f"Validating from {CV_START_YEAR} to {CV_END_YEAR}")

In [ ]:
for path in [DATA_PATH,
             f"/kaggle/input/datasets/adityasai848/artifacts/{DATA_PATH}"]:
    if os.path.exists(path):
        df = pd.read_parquet(path)
        print(f"Loaded: {path}")
        break
else:
    raise FileNotFoundError(
        f"Cannot find {DATA_PATH}. Upload it first."
    )

for path in [FEAT_PATH,
             f"/kaggle/input/datasets/adityasai848/artifacts/{FEAT_PATH}"]:
    if os.path.exists(path):
        with open(path) as f:
            feature_cols = json.load(f)
        break
else:
    raise FileNotFoundError(f"Cannot find {FEAT_PATH}")

for path in [CRISIS_PATH,
             f"/kaggle/input/datasets/adityasai848/artifacts/{CRISIS_PATH}"]:
    if os.path.exists(path):
        with open(path) as f:
            crisis_periods = json.load(f)
        break
else:
    raise FileNotFoundError(f"Cannot find {CRISIS_PATH}")

print(f"DataFrame:  {df.shape}")
print(f"Date range: {df.index[0].date()} -> {df.index[-1].date()}")

In [ ]:
X_raw = df[feature_cols].copy()

# 1. Convert everything to numeric (coerce errors to NaN)
for col in X_raw.columns:
    X_raw[col] = pd.to_numeric(X_raw[col], errors='coerce')

# 2. Replace infinities with NaN
X_raw = X_raw.replace([np.inf, -np.inf], np.nan)

# 3. Fill NaNs with column median, and 0.0 as final fallback
for col in X_raw.columns:
    col_median = X_raw[col].median()
    if pd.isna(col_median):
        col_median = 0.0
    X_raw[col] = X_raw[col].fillna(col_median)
    
# 4. Aggressive final fallback for any bizarre remaining NaNs
X_raw = X_raw.fillna(0.0)

for col in X_raw.columns:
    lo = X_raw[col].quantile(CLIP_LOW)
    hi = X_raw[col].quantile(CLIP_HIGH)
    if pd.isna(lo): lo = 0.0
    if pd.isna(hi): hi = 0.0
    X_raw[col] = X_raw[col].clip(lo, hi)

df_clean = X_raw.copy()
df_clean['crisis_label'] = df['crisis_label'].values

assert not df_clean[feature_cols].isnull().any().any(), "NaN values remain"
print("Global features cleaned successfully.")

In [ ]:
class GRN(nn.Module):
    def __init__(self, in_d, hid_d, out_d, drop=DROPOUT):
        super().__init__()
        self.fc1  = nn.Linear(in_d, hid_d)
        self.fc2  = nn.Linear(hid_d, out_d)
        self.gate = nn.Linear(in_d, out_d)
        self.norm = nn.LayerNorm(out_d)
        self.drop = nn.Dropout(drop)
        self.elu  = nn.ELU()
        self.res  = (nn.Linear(in_d, out_d) if in_d != out_d else nn.Identity())

    def forward(self, x):
        r = self.res(x)
        h = self.drop(self.fc2(self.elu(self.fc1(x))))
        g = torch.sigmoid(self.gate(x))
        return self.norm(g * h + (1 - g) * r)

class VSN(nn.Module):
    def __init__(self, in_d, hid_d, drop=DROPOUT):
        super().__init__()
        self.in_d    = in_d
        self.var_nns = nn.ModuleList([
            nn.Sequential(
                nn.Linear(1, hid_d), nn.ELU(),
                nn.Dropout(drop), nn.Linear(hid_d, hid_d)
            ) for _ in range(in_d)
        ])
        self.wnet = nn.Sequential(
            nn.Linear(in_d, hid_d), nn.ELU(),
            nn.Linear(hid_d, in_d), nn.Softmax(dim=-1)
        )
        self.proj = nn.Linear(hid_d, hid_d)

    def forward(self, x):
        w    = self.wnet(x.mean(dim=1))
        outs = [self.var_nns[i](x[:,:,i:i+1]) for i in range(self.in_d)]
        stk  = torch.stack(outs, dim=-1)
        sel  = (stk * w.unsqueeze(1).unsqueeze(2)).sum(dim=-1)
        return self.proj(sel), w

class CrashSignalTFT(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        H = HIDDEN_DIM
        self.vsn  = VSN(input_dim, H)
        self.lstm = nn.LSTM(
            H, H, LSTM_LAYERS, batch_first=True,
            dropout=DROPOUT if LSTM_LAYERS > 1 else 0
        )
        self.grn1   = GRN(H, H, H)
        self.attn   = nn.MultiheadAttention(H, NUM_HEADS, dropout=DROPOUT, batch_first=True)
        self.grn2   = GRN(H, H, H)
        self.norm   = nn.LayerNorm(H)
        self.clf    = nn.Sequential(
            nn.Linear(H, H//2), nn.ReLU(),
            nn.Dropout(DROPOUT), nn.Linear(H//2, NUM_CLASSES)
        )
        self.stress = nn.Sequential(
            nn.Linear(H, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid()
        )

    def forward(self, x):
        x, w     = self.vsn(x)
        x, _     = self.lstm(x)
        x        = self.grn1(x)
        x2, _    = self.attn(x, x, x)
        x        = self.grn2(x2 + x)
        x        = self.norm(x)
        f        = x[:, -1, :]
        return self.clf(f), self.stress(f) * 100, w

class HybridStressLoss(nn.Module):
    def __init__(self, class_weights):
        super().__init__()
        self.cw = class_weights

    def ce_loss(self, logits, targets):
        return F.cross_entropy(logits, targets, weight=self.cw, label_smoothing=LABEL_SMOOTH)

    def ordinal_loss(self, logits, targets):
        probs = F.softmax(logits, dim=-1)
        loss  = 0.0
        for k in range(NUM_CLASSES - 1):
            p = probs[:, k+1:].sum(dim=-1)
            b = (targets > k).float()
            loss += F.binary_cross_entropy(p.clamp(1e-7, 1-1e-7), b)
        return loss / (NUM_CLASSES - 1)

    def smooth_loss(self, stress):
        if stress.shape[0] < 2: return torch.tensor(0.0, device=stress.device)
        s = stress.squeeze(-1)
        diff = (s[1:] - s[:-1]) / 100.0
        return diff.pow(2).mean()

    def forward(self, logits, stress, targets):
        l_ce  = self.ce_loss(logits, targets)
        l_ord = self.ordinal_loss(logits, targets)
        l_smo = self.smooth_loss(stress)
        return (ALPHA * l_ce + BETA * l_ord + GAMMA * l_smo)

class StressDS(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

def make_sequences(X, y, w):
    Xs = np.array([X[i:i+w] for i in range(len(X)-w)], dtype=np.float32)
    ys = np.array([y[i+w] for i in range(len(X)-w)])
    return Xs, ys

In [ ]:
print("=" * 65)
print("CRASHSIGNAL v4 — WALK-FORWARD VALIDATION")
print("=" * 65)

oof_preds  = []
oof_probs  = []
oof_labels = []
oof_dates  = []
cv_metrics = {}

os.makedirs(OUT_DIR, exist_ok=True)

# Walk-forward loop
for val_year in range(CV_START_YEAR, CV_END_YEAR + 1):
    
    # Reset seeds per fold to ensure independent initialization
    torch.manual_seed(42 + val_year)
    np.random.seed(42 + val_year)
    
    print(f"\n[{val_year}] Preparing fold...")
    
    # 1. Temporal Split
    train_mask = df_clean.index.year < val_year
    val_mask   = df_clean.index.year == val_year
    
    X_train = df_clean.loc[train_mask, feature_cols].values
    X_val   = df_clean.loc[val_mask, feature_cols].values
    y_train = df_clean.loc[train_mask, 'crisis_label'].values
    y_val   = df_clean.loc[val_mask, 'crisis_label'].values
    d_val   = df_clean.index[val_mask]
    
    # 2. Fit Scaler ONLY on train
    scaler = RobustScaler()
    X_tr_s = scaler.fit_transform(X_train)
    X_va_s = scaler.transform(X_val)
    
    # 3. Sequencing
    X_tr_seq, y_tr_seq = make_sequences(X_tr_s, y_train, WINDOW_SIZE)
    X_va_seq, y_va_seq = make_sequences(X_va_s, y_val, WINDOW_SIZE)
    # The actual dates corresponding to the validation labels (shifted by WINDOW_SIZE)
    oof_d = d_val[WINDOW_SIZE:]
    
    # Skip year if validation set becomes too small after windowing
    if len(X_va_seq) == 0:
        print(f"[{val_year}] Skipping — insufficient days for windowing.")
        continue
    
    # 4. Class Weights
    raw_w  = compute_class_weight("balanced", classes=np.unique(y_tr_seq), y=y_tr_seq)
    sqrt_w = np.sqrt(raw_w) / np.sqrt(raw_w).mean()
    cw     = torch.tensor(sqrt_w, dtype=torch.float32).to(device)
    
    tr_loader = DataLoader(StressDS(X_tr_seq, y_tr_seq), batch_size=BATCH_SIZE, shuffle=True)
    va_loader = DataLoader(StressDS(X_va_seq, y_va_seq), batch_size=BATCH_SIZE, shuffle=False)
    
    # 5. Model & Criterion Reset
    model = CrashSignalTFT(len(feature_cols)).to(device)
    criterion = HybridStressLoss(cw)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=COSINE_TMAX, eta_min=COSINE_EMIN)
    
    best_f1   = 0.0
    pat_ctr   = 0
    fold_path = f"{OUT_DIR}/model_{val_year}.pth"
    fold_probs, fold_preds = [], []
    
    # 6. Train Loop
    for ep in range(NUM_EPOCHS):
        model.train()
        for xb, yb in tr_loader:
            xb, yb = xb.to(device), yb.to(device)
            lg, st, _ = model(xb)
            loss = criterion(lg, st, yb)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
        scheduler.step()
        
        model.eval()
        vlb, vp, vpr = [], [], []
        with torch.no_grad():
            for xb, yb in va_loader:
                lg, _, _ = model(xb.to(device))
                probs    = F.softmax(lg, dim=-1)
                vp.extend(lg.argmax(1).cpu().numpy())
                vlb.extend(yb.numpy())
                vpr.extend(probs.cpu().numpy())
        
        va_f1 = f1_score(vlb, vp, average="macro", zero_division=0)
        
        if va_f1 > best_f1:
            best_f1 = va_f1
            pat_ctr = 0
            torch.save(model.state_dict(), fold_path)
            fold_probs = vpr
            fold_preds = vp
        else:
            pat_ctr += 1
            if pat_ctr >= PATIENCE:
                break
                
    print(f"[{val_year}] Finished. Best F1: {best_f1:.4f}")
    
    # 7. Collect OOF Predictions
    if len(fold_preds) > 0:
        oof_preds.extend(fold_preds)
        oof_probs.extend(fold_probs)
        oof_labels.extend(y_va_seq)
        oof_dates.extend(oof_d)
        cv_metrics[val_year] = best_f1
    else:
        print(f"[{val_year}] Failed to converge significantly.")

In [ ]:
print("\n" + "=" * 55)
print("OUT-OF-FOLD (OOF) WALK-FORWARD RESULTS (2016-2024)")
print("=" * 55)

oof_f1  = f1_score(oof_labels, oof_preds, average="macro", zero_division=0)
oof_auc = roc_auc_score(oof_labels, oof_probs, multi_class="ovr", average="macro")

print(f"Honest Global F1:  {oof_f1:.4f}")
print(f"Honest Global AUC: {oof_auc:.4f}")

print("\nClassification Report across all regimes:")
print(classification_report(oof_labels, oof_preds, target_names=["Normal","Stress","Crisis"], digits=4))

cm = confusion_matrix(oof_labels, oof_preds)
fig, ax = plt.subplots(figsize=(7, 5), facecolor="#0a0e1a")
ax.set_facecolor("#0a0e1a")
sns.heatmap(cm, annot=True, fmt="d",
    xticklabels=["Normal","Stress","Crisis"],
    yticklabels=["Normal","Stress","Crisis"],
    cmap="Reds", ax=ax)
ax.set_xlabel("Predicted", color="white")
ax.set_ylabel("Actual",    color="white")
ax.set_title("OOF Confusion Matrix (Walk-Forward)", color="white")
ax.tick_params(colors="white")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/oof_confusion_matrix.png", dpi=120, facecolor="#0a0e1a")
plt.show()

In [ ]:
# ── Train Final Production Model on ALL Data ──────────────────────────────
# We use the median number of epochs and parameters found during walk-forward.

print("\n" + "=" * 55)
print("TRAINING FINAL PRODUCTION MODEL (1994 - 2024)")
print("=" * 55)

X_all = df_clean[feature_cols].values
y_all = df_clean['crisis_label'].values
d_all = df_clean.index

scaler = RobustScaler()
X_all_s = scaler.fit_transform(X_all)

X_all_seq, y_all_seq = make_sequences(X_all_s, y_all, WINDOW_SIZE)

raw_w  = compute_class_weight("balanced", classes=np.unique(y_all_seq), y=y_all_seq)
sqrt_w = np.sqrt(raw_w) / np.sqrt(raw_w).mean()
cw     = torch.tensor(sqrt_w, dtype=torch.float32).to(device)

all_loader = DataLoader(StressDS(X_all_seq, y_all_seq), batch_size=BATCH_SIZE, shuffle=True)

final_model = CrashSignalTFT(len(feature_cols)).to(device)
criterion   = HybridStressLoss(cw)
optimizer   = torch.optim.AdamW(final_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# Target 30 epochs as a safe middle-ground found during early-stopping
FINAL_EPOCHS = 30 
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FINAL_EPOCHS, eta_min=COSINE_EMIN)

final_model.train()
for ep in range(FINAL_EPOCHS):
    for xb, yb in all_loader:
        xb, yb = xb.to(device), yb.to(device)
        lg, st, _ = final_model(xb)
        loss = criterion(lg, st, yb)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(final_model.parameters(), GRAD_CLIP)
        optimizer.step()
    scheduler.step()

final_path = f"{OUT_DIR}/crashsignal_model_final.pth"
torch.save(final_model.state_dict(), final_path)
print("Final production model trained and saved.")

In [ ]:
def gen_scores(model, X, dates, y, w, bs=256):
    model.eval()
    scores, preds, wts = [], [], []
    seqs = np.array([X[i:i+w] for i in range(len(X)-w)], dtype=np.float32)
    for i in tqdm(range(0, len(seqs), bs), desc="Scoring History"):
        b = torch.tensor(seqs[i:i+bs]).to(device)
        with torch.no_grad():
            lg, st, wt = model(b)
        scores.extend(st.squeeze(-1).cpu().numpy().tolist())
        preds.extend(lg.argmax(1).cpu().numpy().tolist())
        wts.append(wt.mean(dim=0).cpu().numpy())

    n   = len(scores)
    out = pd.DataFrame({
        "date":            dates[w:w+n],
        "stress_score":    scores,
        "predicted_label": preds,
        "crisis_label":    y[w:w+n]
    }).set_index("date")
    return out, np.mean(wts, axis=0)

scores_df, avg_wt = gen_scores(final_model, X_all_s, d_all, y_all, WINDOW_SIZE)
print(f"Historical Scores Created: {len(scores_df)} days")

In [ ]:
fig, ax = plt.subplots(figsize=(18, 4), facecolor="#0a0e1a")
ax.set_facecolor("#0a0e1a")
ax.plot(scores_df.index, scores_df["stress_score"], color="#4a9eff", lw=0.7)
for cp in crisis_periods:
    s = pd.Timestamp(cp["start"]); e = pd.Timestamp(cp["end"])
    col = "#ff4444" if cp["severity"]==2 else "#ff8c00"
    ax.axvspan(s, e, alpha=0.25 if cp["severity"]==2 else 0.12, color=col)
ax.set_ylim(0,100)
ax.set_title("CrashSignal — 30 Year Stress History (Production Model)", color="white", fontsize=13)
ax.tick_params(colors="white")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/stress_timeline.png", dpi=120, facecolor="#0a0e1a")
plt.show()

imp_df = pd.DataFrame({
    "indicator":  feature_cols[:len(avg_wt)],
    "importance": avg_wt
}).sort_values("importance", ascending=False)

joblib.dump(scaler, f"{OUT_DIR}/crashsignal_scaler.pkl")
scores_df.reset_index().to_parquet(f"{OUT_DIR}/historical_stress.parquet")
imp_df.to_csv(f"{OUT_DIR}/variable_importance.csv", index=False)

print(f"\n[SUCCESS] OOF Walk-Forward Validation F1: {oof_f1:.4f}")
print("[SUCCESS] Walk-Forward notebook generation absolute success.")